1. Import dan membaca dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import ast

from pathlib import Path
from functools import lru_cache
from IPython.display import display

from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from rank_bm25 import BM25Okapi

# Folder Eksperiment1 berada di dalam folder proyek
ROOT = Path.cwd()
if (ROOT.parent / "queries_dev.csv").exists():
    ROOT = ROOT.parent

# Membaca data yang sudah dibagi
dev_df = pd.read_csv(ROOT / "queries_dev.csv")
test_df = pd.read_csv(ROOT / "queries_test.csv")

# File pendukung
corpus_file = ROOT / "corpus_faq.csv"
if not corpus_file.exists():
    corpus_file = ROOT / "corpus(1).csv"

kamus_file = ROOT / "kamus_normalisasi.csv"
if not kamus_file.exists():
    kamus_file = ROOT / "kamus(1).csv"

corpus = pd.read_csv(corpus_file)
kamus = pd.read_csv(kamus_file)

OUTPUT_DIR = ROOT / "Eksperiment1" / "hasil_exp1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Corpus:", len(corpus))
print("Kamus:", len(kamus))
print("Development set:", len(dev_df))
print("Test set:", len(test_df))

display(dev_df.head())

Corpus: 40
Kamus: 73
Development set: 166
Test set: 499


,query_id,query_text,gaya_kueri,relevant_doc_id,panjang_kata,rasio_tdk_baku,tokens_baseline
0,Q0108,jika mahasiswa menghasilkan karya atau produk ...,formal,FAQ039,16,0.062500,"['mahasiswa', 'hasil', 'karya', 'produk', 'lam..."
1,Q0039,"harusnya saya 3 orang ibu, tapi baru 2 orang d...",formal,FAQ011,19,0.000000,"['harus', '3', 'orang', 'ibu', 'baru', '2', 'o..."
2,Q0451,"Mohon info, dosen pembimbing tidak bisa hadir ...",parafrase,FAQ025,16,0.000000,"['mohon', 'info', 'dosen', 'bimbing', 'hadir',..."
3,Q0590,Seperti apa langkah melakukan pengajuan Lapora...,parafrase,FAQ035,12,0.083333,"['apa', 'langkah', 'laku', 'aju', 'lapor', 'up..."
4,Q0038,teman kelompok ku masih belum join dipengelomp...,informal,FAQ011,12,0.000000,"['teman', 'kelompok', 'ku', 'join', 'kelompok'..."


2. Menggunakan hasil preprocessing baseline

In [2]:
def parse_tokens(value):
    return ast.literal_eval(value)

dev_df["tokens_baseline_list"] = (
    dev_df["tokens_baseline"].apply(parse_tokens)
)

test_df["tokens_baseline_list"] = (
    test_df["tokens_baseline"].apply(parse_tokens)
)

print("Contoh token baseline:")
print(dev_df["tokens_baseline_list"].iloc[0])

Contoh token baseline:
['mahasiswa', 'hasil', 'karya', 'produk', 'lama', 'pi', 'potensi', 'daftar', 'hki', 'apa', 'perlu', 'laku']


3. Membangun indeks BM25

In [3]:
stemmer = StemmerFactory().create_stemmer()

stopwords = set(
    StopWordRemoverFactory().get_stop_words()
)

@lru_cache(maxsize=None)
def stem_cached(kata):
    return stemmer.stem(kata)

def tokenisasi_dasar(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

def proses_token(tokens):
    tokens = [
        token for token in tokens
        if token not in stopwords
    ]

    return [stem_cached(token) for token in tokens]

def preprocess_corpus(text):
    return proses_token(tokenisasi_dasar(text))

corpus["teks_indeks"] = (
    corpus["pertanyaan_baku"].astype(str)
    + " "
    + corpus["jawaban"].astype(str)
)

corpus_tokens = corpus["teks_indeks"].apply(
    preprocess_corpus
).tolist()

doc_ids = corpus["doc_id"].tolist()

# Sama dengan baseline
K1 = 1.5
B = 0.75

bm25 = BM25Okapi(
    corpus_tokens,
    k1=K1,
    b=B
)

print("Indeks BM25 berhasil dibangun")
print("Jumlah dokumen:", len(doc_ids))
print("Rata-rata panjang dokumen:", bm25.avgdl)

Indeks BM25 berhasil dibangun
Jumlah dokumen: 40
Rata-rata panjang dokumen: 26.225


4. Membuat fungsi normalisasi kamus

In [4]:
# Menghindari padanan kosong dan ambigu
kamus_valid = kamus.dropna(
    subset=["tidak_baku", "baku"]
).copy()

kamus_valid = kamus_valid[
    ~kamus_valid["baku"].astype(str).str.contains(
        "/", regex=False
    )
]

pasangan_kamus = []

for _, row in kamus_valid.iterrows():

    asal = tuple(
        tokenisasi_dasar(row["tidak_baku"])
    )

    target = tokenisasi_dasar(row["baku"])

    if asal and target:
        pasangan_kamus.append(
            (asal, target)
        )

# Frasa terpanjang diperiksa terlebih dahulu
pasangan_kamus.sort(
    key=lambda x: len(x[0]),
    reverse=True
)

def normalisasi_kamus(text):

    tokens = tokenisasi_dasar(text)

    hasil = []
    perubahan = []
    i = 0

    while i < len(tokens):

        ditemukan = False

        for asal, target in pasangan_kamus:

            panjang = len(asal)

            if tuple(tokens[i:i+panjang]) == asal:

                hasil.extend(target)

                perubahan.append(
                    f'{" ".join(asal)} -> {" ".join(target)}'
                )

                i += panjang
                ditemukan = True
                break

        if not ditemukan:
            hasil.append(tokens[i])
            i += 1

    # Stopword removal dan stemming dilakukan
    # setelah normalisasi kamus
    tokens_final = proses_token(hasil)

    return tokens_final, perubahan


# Contoh penggunaan
contoh = "dospem blm verif berkas pi, gmn caranya?"

tokens, perubahan = normalisasi_kamus(contoh)

print("Kalimat:", contoh)
print("Hasil token:", tokens)
print("Perubahan:", perubahan)

Kalimat: dospem blm verif berkas pi, gmn caranya?
Hasil token: ['dosen', 'bimbing', 'verifikasi', 'berkas', 'praktik', 'industri', 'bagaimana', 'cara']
Perubahan: ['dospem -> dosen pembimbing', 'blm -> belum', 'verif -> verifikasi', 'pi -> praktik industri', 'gmn -> bagaimana']


5. Fungsi evaluasi BM25

In [5]:
def evaluasi_per_kueri(data):

    hasil = []

    for row in data.itertuples(index=False):

        # Baseline: token tersimpan dari CSV
        token_base = row.tokens_baseline_list

        # Eksperimen 1: normalisasi kamus
        token_exp1, perubahan = normalisasi_kamus(
            row.query_text
        )

        # Hitung skor relevansi
        skor_base = bm25.get_scores(token_base)
        skor_exp1 = bm25.get_scores(token_exp1)

        # Peringkat dokumen
        idx_base = np.argsort(skor_base)[::-1]
        idx_exp1 = np.argsort(skor_exp1)[::-1]

        ranking_base = [
            doc_ids[i] for i in idx_base
        ]

        ranking_exp1 = [
            doc_ids[i] for i in idx_exp1
        ]

        # Dokumen benar
        relevan = row.relevant_doc_id

        rank_base = ranking_base.index(relevan) + 1
        rank_exp1 = ranking_exp1.index(relevan) + 1

        hasil.append({

            "query_id": row.query_id,
            "gaya_kueri": row.gaya_kueri,
            "query_text": row.query_text,

            "relevant_doc_id": relevan,

            "normalisasi": " | ".join(perubahan),
            "jumlah_perubahan": len(perubahan),

            "prediksi_baseline": ranking_base[0],
            "prediksi_exp1": ranking_exp1[0],

            "rank_baseline": rank_base,
            "rank_exp1": rank_exp1,

            "hit_baseline": int(rank_base == 1),
            "hit_exp1": int(rank_exp1 == 1),

            "rr_baseline": 1 / rank_base,
            "rr_exp1": 1 / rank_exp1,

            "score_baseline": float(
                skor_base[idx_base[0]]
            ),

            "score_exp1": float(
                skor_exp1[idx_exp1[0]]
            )
        })

    return pd.DataFrame(hasil)

6. Menghitung Hit@1, MRR, dan Fallback Rate

In [6]:
THRESHOLD_BM25 = 3

def hitung_metrik(hasil, konfigurasi):

    hit = hasil[f"hit_{konfigurasi}"].mean()
    mrr = hasil[f"rr_{konfigurasi}"].mean()

    skor = hasil[f"score_{konfigurasi}"]

    if THRESHOLD_BM25 is None:
        fallback = np.nan
    else:
        fallback = (
            skor < THRESHOLD_BM25
        ).mean()

    return {
        "N": len(hasil),
        "Hit@1": hit,
        "MRR": mrr,
        "Fallback Rate": fallback
    }


# Evaluasi development set
hasil_dev = evaluasi_per_kueri(dev_df)

baseline = hitung_metrik(
    hasil_dev, "baseline"
)

eksperimen1 = hitung_metrik(
    hasil_dev, "exp1"
)

perbandingan = pd.DataFrame({
    "BM25 Baseline": baseline,
    "BM25 + Kamus": eksperimen1
}).T

print("=== HASIL EVALUASI DEV ===")

display(perbandingan.round(4))

=== HASIL EVALUASI DEV ===


,N,Hit@1,MRR,Fallback Rate
BM25 Baseline,166.0,0.9398,0.9620,0.006
BM25 + Kamus,166.0,0.9096,0.9443,0.006


7.  Evaluasi berdasarkan lima gaya kueri

In [7]:
gaya_kueri = [
    "formal",
    "informal",
    "singkatan",
    "typo",
    "parafrase"
]

hasil_per_gaya = []

for gaya in gaya_kueri:

    subset = hasil_dev[
        hasil_dev["gaya_kueri"] == gaya
    ]

    baseline_gaya = hitung_metrik(
        subset, "baseline"
    )

    exp1_gaya = hitung_metrik(
        subset, "exp1"
    )

    hasil_per_gaya.append({

        "Gaya Kueri": gaya,
        "Jumlah": len(subset),

        "Hit@1 Baseline": baseline_gaya["Hit@1"],
        "Hit@1 Exp1": exp1_gaya["Hit@1"],

        "MRR Baseline": baseline_gaya["MRR"],
        "MRR Exp1": exp1_gaya["MRR"],

        "Delta Hit@1": (
            exp1_gaya["Hit@1"]
            - baseline_gaya["Hit@1"]
        ),

        "Delta MRR": (
            exp1_gaya["MRR"]
            - baseline_gaya["MRR"]
        )
    })

per_gaya_df = pd.DataFrame(hasil_per_gaya)

display(per_gaya_df.round(4))

,Gaya Kueri,Jumlah,Hit@1 Baseline,Hit@1 Exp1,MRR Baseline,MRR Exp1,Delta Hit@1,Delta MRR
0,formal,20,1.0000,0.8500,1.0000,0.9167,-0.1500,-0.0833
1,informal,36,0.9722,0.9722,0.9861,0.9861,0.0000,0.0000
2,singkatan,35,0.9143,0.9143,0.9476,0.9524,0.0000,0.0048
3,typo,40,0.8750,0.8750,0.9132,0.9140,0.0000,0.0008
4,parafrase,35,0.9714,0.9143,0.9857,0.9438,-0.0571,-0.0419


9. Evaluasi test set pada tahap akhir

In [8]:
hasil_dev.to_csv(
    OUTPUT_DIR / "hasil_exp1_dev.csv",
    index=False
)

perbandingan.to_csv(
    OUTPUT_DIR / "evaluasi_exp1_dev.csv"
)

per_gaya_df.to_csv(
    OUTPUT_DIR / "evaluasi_per_gaya_dev.csv",
    index=False
)

print("Hasil evaluasi berhasil disimpan!")
print("Lokasi:", OUTPUT_DIR)

Hasil evaluasi berhasil disimpan!
Lokasi: d:\Folder KULIAH\Sistem Temu Kembali\Artikel (1)\Eksperiment1\hasil_exp1
